# Historical Medical Data Aug notebook — source only
All 24 original cell sources follow unchanged. Outputs, execution counts, attachments and metadata were removed; no CT/label data is included.
This is an audit/reference notebook, NOT the supported training entry point. Do not Run All: original cells include permissive shape cropping, the batch uint8-distance bug, independent configurations and NIfTI writes.
Use ../../tools/run_feedback_experiment.py and ../../docs/cp_input_repair.md for the current integrated pipeline. The initial guard intentionally prevents accidental Run All.


In [ ]:
raise RuntimeError("Historical source notebook; review cells only. Use tools/run_feedback_experiment.py for supported training.")


In [ ]:
import os
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt

In [ ]:
# ---------- CONFIG ----------
# Point these to your files
IMAGE_PATH = "Data/image/liver_1_0000.nii.gz"
LABEL_PATH = "Data/labels/liver_1.nii.gz"

In [ ]:
# Slice indices to visualize (set to 'mid' to auto-pick middle)
SLICE_IDX = {"axial": "mid", "coronal": "mid", "sagittal": "mid"}
LABEL_ALPHA = 0.35  # overlay transparency

In [ ]:
# ---------- UTILS ----------
def load_nifti(path):
    ni = nib.load(path)
    data = ni.get_fdata(dtype=np.float32)  # float32 for math + display
    hdr = ni.header
    affine = ni.affine
    spacing = hdr.get_zooms()[: data.ndim]  # voxel spacing per axis
    return data, affine, spacing, hdr

def ensure_3d(vol):
    """
    Many medical datasets store images as:
      - 3D: (Z, Y, X)
      - 4D: (C, Z, Y, X) or (Z, Y, X, C)
    We’ll reduce to (Z, Y, X), picking the first channel if needed.
    """
    if vol.ndim == 3:
        return vol
    if vol.ndim == 4:
        # Try (C, Z, Y, X)
        if vol.shape[0] <= 4 and vol.shape[1] >= 8:
            return vol[0]
        # Try (Z, Y, X, C)
        if vol.shape[-1] <= 4 and vol.shape[0] >= 8:
            return vol[..., 0]
    raise ValueError(f"Unsupported shape {vol.shape}; expected 3D or 4D with a small channel dim.")

def pick_index(length, spec):
    if spec == "mid":
        return length // 2
    if isinstance(spec, int):
        return max(0, min(length - 1, spec))
    raise ValueError("Slice spec must be 'mid' or an int.")

def normalize_for_display(vol, p_low=1, p_high=99):
    """
    Robust min-max normalization using percentiles to avoid outliers.
    Returns float64 in [0,1].
    """
    low = np.percentile(vol, p_low)
    high = np.percentile(vol, p_high)
    if high <= low:
        high = low + 1e-6
    vol = np.clip(vol, low, high)
    return (vol - low) / (high - low)


In [ ]:
# ---------- LOAD ----------
img_raw, img_affine, img_spacing, img_hdr = load_nifti(IMAGE_PATH)
lab_raw, lab_affine, lab_spacing, lab_hdr = load_nifti(LABEL_PATH)

img = ensure_3d(img_raw)
lab = ensure_3d(lab_raw)

In [ ]:
# sanity check on spatial dims (allow small mismatches and crop if needed)
minZ = min(img.shape[0], lab.shape[0])
minY = min(img.shape[1], lab.shape[1])
minX = min(img.shape[2], lab.shape[2])
if (minZ, minY, minX) != img.shape or (minZ, minY, minX) != lab.shape:
    img = img[:minZ, :minY, :minX]
    lab = lab[:minZ, :minY, :minX]

In [ ]:
# ---------- INFO ----------
print("Image shape (Z,Y,X):", img.shape, "spacing:", img_spacing[:3])
print("Label shape (Z,Y,X):", lab.shape, "spacing:", lab_spacing[:3])
print("Image dtype:", img_raw.dtype, "| Label dtype:", lab_raw.dtype)
unique_labels = np.unique(lab.astype(np.int32))
print("Unique label values:", unique_labels)

In [ ]:
# ---------- DISPLAY SLICES ----------
# Normalize image for display
img_disp = normalize_for_display(img)

In [ ]:
# Choose indices
iz = pick_index(img.shape[0], SLICE_IDX["axial"])
iy = pick_index(img.shape[1], SLICE_IDX["coronal"])
ix = pick_index(img.shape[2], SLICE_IDX["sagittal"])

In [ ]:
# Extract planes
axial_img, axial_lab = img_disp[iz], lab[iz]
coronal_img, coronal_lab = img_disp[:, iy, :], lab[:, iy, :]
sagittal_img, sagittal_lab = img_disp[:, :, ix], lab[:, :, ix]

In [ ]:
# Plot 3 views with label overlay
fig = plt.figure(figsize=(12, 10))

# Axial (Z fixed) -> plane is (Y,X)
plt.subplot(2, 2, 1)
plt.title(f"Axial (z={iz})")
plt.imshow(axial_img, origin="lower")
# Overlay labels (non-zero as foreground). Using default colormap with alpha.
plt.imshow((axial_lab > 0).astype(float), origin="lower", alpha=LABEL_ALPHA)
plt.axis("off")

In [ ]:
# Plot 3 views with label overlay
fig = plt.figure(figsize=(12, 10))

# Axial (Z fixed) -> plane is (Y,X)
plt.subplot(2, 2, 1)
plt.title(f"Axial (z={iz})")
plt.imshow(axial_img, origin="lower")
# Overlay labels (non-zero as foreground). Using default colormap with alpha.
plt.imshow((axial_lab > 0).astype(float), origin="lower", alpha=LABEL_ALPHA)
plt.axis("off")

In [ ]:
# Sagittal (X fixed) -> plane is (Z,Y)
plt.subplot(2, 2, 3)
plt.title(f"Sagittal (x={ix})")
plt.imshow(sagittal_img.T, origin="lower")   # transpose so (Z,Y) reads left-right
plt.imshow((sagittal_lab > 0).astype(float).T, origin="lower", alpha=LABEL_ALPHA)
plt.axis("off")

In [ ]:
plt.tight_layout()
plt.show()


In [ ]:
voxel_count = np.prod(img.shape)
label_voxel_frac = (lab > 0).sum() / voxel_count
print(f"Foreground (label>0) voxels: {label_voxel_frac:.4%}")

In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure

# ---- Paths ----
img_path = "Data/image/liver_1_0000.nii.gz"
lab_path = "Data/labels/liver_1.nii.gz"

# ---- Load ----
img = nib.load(img_path).get_fdata()
lab = nib.load(lab_path).get_fdata()

# Pick the structure you want (e.g., liver=1, tumor=2, etc.)
target_label = 1
mask = (lab == target_label).astype(np.uint8)

# ---- Extract 3D surface (marching cubes) ----
verts, faces, normals, values = measure.marching_cubes(mask, level=0.5)

# ---- Plot ----
fig = plt.figure(figsize=(8, 8))
ax = fig.add_subplot(111, projection="3d")

# Surface
mesh = ax.plot_trisurf(
    verts[:, 0], verts[:, 1], faces, verts[:, 2],
    cmap="Spectral", lw=0.5, alpha=0.8
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("3D Liver Surface")

plt.show()


In [ ]:
import nibabel as nib
import numpy as np
import plotly.graph_objects as go

lab = nib.load("Data/labels/liver_1.nii.gz").get_fdata()
mask = (lab == 1)  # target structure

fig = go.Figure(data=go.Volume(
    x=np.arange(mask.shape[0]).repeat(mask.shape[1]*mask.shape[2]),
    y=np.tile(np.arange(mask.shape[1]).repeat(mask.shape[2]), mask.shape[0]),
    z=np.tile(np.arange(mask.shape[2]), mask.shape[0]*mask.shape[1]),
    value=mask.flatten().astype(np.float32),
    isomin=0.5, isomax=1.0,
    opacity=0.1,   # low for volume, increase for dense
    surface_count=1
))
fig.show()


In [ ]:
import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from skimage import measure

# ---- Load data ----
img = nib.load("Data/image/liver_1_0000.nii.gz").get_fdata()
lab = nib.load("Data/labels/liver_1.nii.gz").get_fdata()

# Normalize image for surface extraction
img_norm = (img - img.min()) / (img.max() - img.min())

# Liver mask (label==1)
mask = (lab == 1).astype(np.uint8)

# ---- Extract surfaces ----
# CT iso-surface (choose intensity ~0.5 for mid-gray structure)
verts_img, faces_img, _, _ = measure.marching_cubes(img_norm, level=0.5, step_size=2)

# Segmentation surface (label=1)
verts_lab, faces_lab, _, _ = measure.marching_cubes(mask, level=0.5)

# ---- Plot ----
fig = plt.figure(figsize=(10, 10))
ax = fig.add_subplot(111, projection="3d")

# Grayscale CT surface
ax.plot_trisurf(
    verts_img[:, 0], verts_img[:, 1], faces_img, verts_img[:, 2],
    color="lightgray", alpha=0.3, linewidth=0
)

# Blue segmentation surface
ax.plot_trisurf(
    verts_lab[:, 0], verts_lab[:, 1], faces_lab, verts_lab[:, 2],
    color="blue", alpha=0.6, linewidth=0
)

ax.set_xlabel("X")
ax.set_ylabel("Y")
ax.set_zlabel("Z")
ax.set_title("3D Liver (Gray CT + Blue Label)")

plt.show()


In [ ]:
import nibabel as nib
import numpy as np

# Load segmentation
lab_path = "Data/labels/liver_1.nii.gz"
lab = nib.load(lab_path).get_fdata().astype(int)

# Get unique labels
unique_labels = np.unique(lab)
print("Unique label values:", unique_labels)

# If you want counts for each label:
for val in unique_labels:
    count = np.sum(lab == val)
    print(f"Label {val}: {count} voxels")


In [ ]:
import nibabel as nib
import numpy as np
from skimage import measure
import plotly.graph_objects as go

# --------- CONFIG ---------
image_path = "Data/image/liver_70_0000.nii.gz"
label_path = "Data/labels/liver_70.nii.gz"
LIVER_LABEL = 1
TUMOR_LABEL = 2
show_ct_volume = False   # True = add semi-transparent CT volume (heavier)
downsample = 2           # used only when show_ct_volume=True (>=2 recommended)

# --------- LOAD ---------
img_nii = nib.load(image_path)
lab_nii = nib.load(label_path)

img = img_nii.get_fdata()          # (Z,Y,X) or similar
lab = lab_nii.get_fdata().astype(int)

# Handle 4D inputs by picking first channel
if img.ndim == 4:
    # assume either (C,Z,Y,X) or (Z,Y,X,C)
    img = img[0] if img.shape[0] <= 4 else img[..., 0]

# Voxel spacing (zooms) — use first 3 dims
zooms = img_nii.header.get_zooms()
spacing = np.array(zooms[:3], dtype=float)  # (Z,Y,X) spacing in mm (or dataset units)

# --------- SURFACES (marching cubes) ---------
# --------- SURFACES (marching cubes with affine) ---------
affine = lab_nii.affine  # 4x4 matrix mapping voxel -> world (mm)

def marching_with_affine(mask, affine):
    v, f, _, _ = measure.marching_cubes(mask, level=0.5)
    v_h = np.c_[v, np.ones(len(v))]           # make homogeneous (N,4)
    v_w = (affine @ v_h.T).T[:, :3]           # apply affine → world coords
    return v_w, f

# Liver
liver_mask = (lab == LIVER_LABEL).astype(np.uint8)
v_liver, f_liver = marching_with_affine(liver_mask, affine) if liver_mask.any() else (np.empty((0,3)), np.empty((0,3),dtype=int))

# Tumor
tumor_mask = (lab == TUMOR_LABEL).astype(np.uint8)
v_tumor, f_tumor = marching_with_affine(tumor_mask, affine) if tumor_mask.any() else (np.empty((0,3)), np.empty((0,3),dtype=int))


# --------- BUILD FIGURE ---------
fig = go.Figure()

# Liver mesh (light gray)
if len(v_liver):
    fig.add_trace(go.Mesh3d(
        x=v_liver[:,0], y=v_liver[:,1], z=v_liver[:,2],
        i=f_liver[:,0], j=f_liver[:,1], k=f_liver[:,2],
        color="lightgray", opacity=0.25, name="Liver", lighting=dict(ambient=0.6)
    ))

# Tumor mesh (blue)
if len(v_tumor):
    fig.add_trace(go.Mesh3d(
        x=v_tumor[:,0], y=v_tumor[:,1], z=v_tumor[:,2],
        i=f_tumor[:,0], j=f_tumor[:,1], k=f_tumor[:,2],
        color="blue", opacity=0.85, name="Tumor", lighting=dict(ambient=0.5)
    ))

# Optional: add semi-transparent CT volume for context
if show_ct_volume:
    # Downsample volume for performance
    z_slice = slice(None, None, downsample)
    y_slice = slice(None, None, downsample)
    x_slice = slice(None, None, downsample)
    vol = img[z_slice, y_slice, x_slice]

    # Normalize intensities robustly (1–99 percentile)
    p1, p99 = np.percentile(vol, [1, 99])
    vol = np.clip(vol, p1, p99)
    vol = (vol - p1) / max(p99 - p1, 1e-6)

    # Build coordinate grid in *physical units* using spacing
    zz, yy, xx = np.mgrid[
        0:vol.shape[0]*spacing[0]:spacing[0],
        0:vol.shape[1]*spacing[1]:spacing[1],
        0:vol.shape[2]*spacing[2]:spacing[2],
    ]
    # Plotly Volume expects 1D arrays for x,y,z
    fig.add_trace(go.Volume(
        x=xx.flatten(), y=yy.flatten(), z=zz.flatten(),
        value=vol.astype(np.float32).flatten(),
        opacity=0.08, surface_count=10, name="CT volume",
        opacityscale=[[0, 0.0], [1, 1.0]],
        showscale=False
    ))

# Layout: equal aspect in data units; nice camera
fig.update_layout(
    scene=dict(
        xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
        aspectmode="data",
    ),
    legend=dict(itemsizing="constant"),
    margin=dict(l=0, r=0, t=30, b=0),
    title="Interactive 3D — Liver (gray) & Tumor (blue)"
)

fig.show()


In [ ]:
import nibabel as nib
import numpy as np
from skimage import measure
from scipy import ndimage as ndi
import plotly.graph_objects as go

# =========================================
# --------- CONFIG (edit as needed) -------
# =========================================
image_path = "Data/image/liver_70_0000.nii.gz"
label_path = "Data/labels/liver_70.nii.gz"
LIVER_LABEL = 1
TUMOR_LABEL = 2
NUM_COPIES = 1                 # how many new tumors to paste
BLEND_BORDER = 0               # voxels to feather tumor edges (0 = hard paste)
INTENSITY_SCALE_RANGE = (0.95, 1.05)
INTENSITY_SHIFT_RANGE = (-5.0, 5.0)
MIN_LIVER_COVERAGE = 0.85      # fraction of pasted tumor that must lie inside liver
AVOID_OVERLAP_WITH_ORIG = True # discourage placing new tumor too close to original
# Keep pasted tumors away from existing tumors & each other
OCCUPIED_CLEARANCE_VOX = 2    # dilated safety margin (voxels) around occupied areas
MIN_CENTER_SEPARATION_VOX = 12  # optional extra spacing between centers
RNG_SEED = None                  # None => non-deterministic; or set an int for reproducible but varied runs

# Visualization
show_ct_volume = False
downsample = 2

# =========================================
# --------- LOAD --------------------------
# =========================================
img_nii = nib.load(image_path)
lab_nii = nib.load(label_path)

img = img_nii.get_fdata()          # (Z,Y,X)
lab = lab_nii.get_fdata().astype(int)

# Handle 4D inputs by picking first channel
if img.ndim == 4:
    img = img[0] if img.shape[0] <= 4 else img[..., 0]

# Spacing / affine
zooms = img_nii.header.get_zooms()
spacing = np.array(zooms[:3], dtype=float)   # (Z,Y,X) spacing in mm
voxel_mm3 = float(spacing[0] * spacing[1] * spacing[2])
affine = lab_nii.affine

# Masks
liver_mask = (lab == LIVER_LABEL)
tumor_mask = (lab == TUMOR_LABEL)

if not liver_mask.any():
    raise ValueError("No liver voxels found (LIVER_LABEL).")
if not tumor_mask.any():
    raise ValueError("No tumor voxels found (TUMOR_LABEL).")

# =========================================
# --------- HELPERS -----------------------
# =========================================
rng = np.random.default_rng(RNG_SEED)

def bbox_of_mask(mask, pad=2):
    zz, yy, xx = np.where(mask)
    zmin, zmax = max(0, zz.min()-pad), min(mask.shape[0]-1, zz.max()+pad)
    ymin, ymax = max(0, yy.min()-pad), min(mask.shape[1]-1, yy.max()+pad)
    xmin, xmax = max(0, xx.min()-pad), min(mask.shape[2]-1, xx.max()+pad)
    return (slice(zmin, zmax+1), slice(ymin, ymax+1), slice(xmin, xmax+1))

def marching_with_affine(mask, affine):
    if not mask.any():
        return np.empty((0,3)), np.empty((0,3), int)
    v, f, _, _ = measure.marching_cubes(mask.astype(np.uint8), level=0.5)
    v_h = np.c_[v, np.ones(len(v))]
    v_w = (affine @ v_h.T).T[:, :3]
    return v_w, f

def feather_alpha(binary_mask, blend_border=3):
    if blend_border <= 0:
        return binary_mask.astype(np.float32)
    dist_to_bg = ndi.distance_transform_edt(binary_mask)
    alpha = np.clip(dist_to_bg / float(blend_border), 0.0, 1.0).astype(np.float32)
    return alpha

def centroid_world(mask):
    if not mask.any():
        return (np.nan, np.nan, np.nan)
    coords = np.column_stack(np.where(mask))
    # voxel-space centroid (z,y,x)
    c_vox = coords.mean(axis=0)
    c_h = np.r_[c_vox, 1.0]
    c_w = (affine @ c_h)[:3]
    return tuple(map(float, c_w))

'''
# Pick a single tumor component (largest)
lab_cc, n_cc = ndi.label(tumor_mask)
sizes = [(i, (lab_cc == i).sum()) for i in range(1, n_cc+1)]
if not sizes:
    raise ValueError("No connected components in tumor mask.")
src_id = max(sizes, key=lambda t: t[1])[0]
src_comp = (lab_cc == src_id)
'''
lab_cc, n_cc = ndi.label(tumor_mask)
if n_cc == 0:
    raise ValueError("No tumor components in mask.")

# --- Option 1: choose a random tumor (all components equally likely)
src_id = np.random.default_rng(RNG_SEED if RNG_SEED is not None else None).integers(1, n_cc + 1)

# --- Option 2 (alternative): choose proportional to size
# sizes = [(i, (lab_cc == i).sum()) for i in range(1, n_cc+1)]
# ids, weights = zip(*sizes)
# src_id = rng.choice(ids, p=np.array(weights)/sum(weights))
src_comp = (lab_cc == src_id)

src_slz, src_sly, src_slx = bbox_of_mask(src_comp, pad=2)
tumor_ct_patch = img[src_slz, src_sly, src_slx].astype(np.float32)
tumor_mask_patch = src_comp[src_slz, src_sly, src_slx].astype(bool)
alpha_patch = feather_alpha(tumor_mask_patch, BLEND_BORDER)

# Distance to original tumor to avoid overlapping (optional)
if AVOID_OVERLAP_WITH_ORIG:
    dt_from_orig = ndi.distance_transform_edt(~tumor_mask)

# =========================================
# --------- PASTE INSIDE SAME VOLUME ------
# =========================================
out_img = img.copy()
out_lab = lab.copy()

# start with existing tumors as "occupied"
occupied_mask = tumor_mask.copy()

# Precompute a dilated "forbidden" zone around occupied voxels
# (keeps patches away by OCCUPIED_CLEARANCE_VOX voxels)
structure = ndi.generate_binary_structure(3, 1)  # 6-neighborhood
forbidden = ndi.binary_dilation(
    occupied_mask,
    structure=structure,
    iterations=int(max(OCCUPIED_CLEARANCE_VOX, 0))
)
dist_from_occupied = ndi.distance_transform_edt(~occupied_mask)


pasted_masks = []   # store each pasted tumor mask (full-volume)
report_rows = []

def try_random_centers_shuffled(liver_mask, patch_shape, rng, max_tries=4000):
    pz, py, px = patch_shape
    hz, hy, hx = pz//2, py//2, px//2
    Z, Y, X = liver_mask.shape
    zmin, zmax = hz, Z - (pz - hz)
    ymin, ymax = hy, Y - (py - hy)
    xmin, xmax = hx, X - (px - hx)
    if zmax <= zmin or ymax <= ymin or xmax <= xmin:
        return
    zz, yy, xx = np.where(liver_mask)
    keep = (zz >= zmin) & (zz < zmax) & (yy >= ymin) & (yy < ymax) & (xx >= xmin) & (xx < xmax)
    cand = np.column_stack([zz[keep], yy[keep], xx[keep]])
    if len(cand) == 0:
        return
    for idx in rng.permutation(len(cand))[:max_tries]:
        cz, cy, cx = cand[idx]
        yield int(cz), int(cy), int(cx)


for k in range(NUM_COPIES):
    placed = False
    for center in try_random_centers_shuffled(liver_mask, tumor_mask_patch.shape, rng, max_tries=4000):
        pz, py, px = tumor_mask_patch.shape
        hz, hy, hx = pz//2, py//2, px//2
        cz, cy, cx = center
        z1, z2 = cz-hz, cz-hz+pz
        y1, y2 = cy-hy, cy-hy+py
        x1, x2 = cx-hx, cx-hx+px

        roi_liver = liver_mask[z1:z2, y1:y2, x1:x2]
        roi_forbidden = forbidden[z1:z2, y1:y2, x1:x2]

        # ---- HARD NO-OVERLAP RULE with occupied (+clearance)
        if (tumor_mask_patch & roi_forbidden).any():
            continue

        # ---- Coverage rule (after forbidding overlap)
        inside_frac = (tumor_mask_patch & roi_liver).sum() / (tumor_mask_patch.sum() + 1e-6)
        if inside_frac < MIN_LIVER_COVERAGE:
            continue

        # ---- Optional: center separation (vs. original+pastes)
        if MIN_CENTER_SEPARATION_VOX > 0:
            if dist_from_occupied[cz, cy, cx] < MIN_CENTER_SEPARATION_VOX:
                continue

        # ---- Paste (intensity jitter + feather)
        scale = rng.uniform(*INTENSITY_SCALE_RANGE)
        shift = rng.uniform(*INTENSITY_SHIFT_RANGE)
        roi_ct = out_img[z1:z2, y1:y2, x1:x2]
        alpha_eff = alpha_patch * tumor_mask_patch.astype(np.float32)
        pasted_ct = (1.0 - alpha_eff) * roi_ct + alpha_eff * (tumor_ct_patch * scale + shift)
        out_img[z1:z2, y1:y2, x1:x2] = pasted_ct

        # ---- Labels
        new_mask_full = np.zeros_like(out_lab, dtype=bool)
        new_mask_full[z1:z2, y1:y2, x1:x2] = tumor_mask_patch
        out_lab[new_mask_full] = TUMOR_LABEL
        pasted_masks.append(new_mask_full)

        # ---- UPDATE OCCUPIED + FORBIDDEN + DISTANCE for next iterations
        occupied_mask |= new_mask_full
        forbidden = ndi.binary_dilation(
            occupied_mask,
            structure=structure,
            iterations=int(max(OCCUPIED_CLEARANCE_VOX, 0))
        )
        dist_from_occupied = ndi.distance_transform_edt(~occupied_mask)

        # ---- reporting (unchanged)
        vox = int(new_mask_full.sum())
        vol_mm3 = vox * voxel_mm3
        c_src = centroid_world(src_comp)
        c_new = centroid_world(new_mask_full)
        offset = tuple(float(b - a) for a, b in zip(c_src, c_new))
        report_rows.append({
            "copy_idx": k+1,
            "voxels": vox,
            "volume_mm3": vol_mm3,
            "center_world_mm": c_new,
            "offset_from_src_mm": offset,
            "liver_coverage": float(inside_frac),
            "scale": float(scale),
            "shift": float(shift),
        })
        placed = True
        break

    if not placed:
        print(f"[Warn] Could not place tumor copy #{k+1} (relax coverage or clearance).")


# =========================================
# --------- PRINT RESULTS -----------------
# =========================================
orig_vox = int(src_comp.sum())
orig_vol = orig_vox * voxel_mm3
print("Original tumor:")
print(f"  voxels: {orig_vox:,}  volume: {orig_vol:,.1f} mm³  center(world): {centroid_world(src_comp)}")
print("\nPasted tumors:")
if not report_rows:
    print("  (none)")
else:
    for r in report_rows:
        c = r["center_world_mm"]
        o = r["offset_from_src_mm"]
        print(f"  #{r['copy_idx']:>2}  voxels={r['voxels']:,}  vol={r['volume_mm3']:,.1f} mm³"
              f"  center={tuple(round(v,1) for v in c)} mm"
              f"  offset={tuple(round(v,1) for v in o)} mm"
              f"  coverage={r['liver_coverage']:.2f}  scale={r['scale']:.3f} shift={r['shift']:.1f}")

# =========================================
# --------- 3D SURFACES -------------------
# =========================================
# Marching cubes for liver, original tumor, pasted tumors
v_liver, f_liver = marching_with_affine(liver_mask, affine)
v_tumor_orig, f_tumor_orig = marching_with_affine(src_comp, affine)

# union of pasted tumors for display
if pasted_masks:
    pasted_union = np.logical_or.reduce(pasted_masks)
else:
    pasted_union = np.zeros_like(lab, bool)

v_tumor_paste, f_tumor_paste = marching_with_affine(pasted_union, affine)

# --------- BUILD FIGURE ---------
fig = go.Figure()

# Liver (light gray)
if len(v_liver):
    fig.add_trace(go.Mesh3d(
        x=v_liver[:,0], y=v_liver[:,1], z=v_liver[:,2],
        i=f_liver[:,0], j=f_liver[:,1], k=f_liver[:,2],
        color="lightgray", opacity=0.25, name="Liver", lighting=dict(ambient=0.6)
    ))

# Original tumor (blue)
if len(v_tumor_orig):
    fig.add_trace(go.Mesh3d(
        x=v_tumor_orig[:,0], y=v_tumor_orig[:,1], z=v_tumor_orig[:,2],
        i=f_tumor_orig[:,0], j=f_tumor_orig[:,1], k=f_tumor_orig[:,2],
        color="blue", opacity=0.85, name="Tumor (original)", lighting=dict(ambient=0.5)
    ))

# Pasted tumor(s) (red)
if len(v_tumor_paste):
    fig.add_trace(go.Mesh3d(
        x=v_tumor_paste[:,0], y=v_tumor_paste[:,1], z=v_tumor_paste[:,2],
        i=f_tumor_paste[:,0], j=f_tumor_paste[:,1], k=f_tumor_paste[:,2],
        color="red", opacity=0.85, name="Tumor (pasted)", lighting=dict(ambient=0.5)
    ))

# Centroid markers
orig_c = centroid_world(src_comp)
fig.add_trace(go.Scatter3d(
    x=[orig_c[0]], y=[orig_c[1]], z=[orig_c[2]],
    mode="markers+text", text=["orig"], textposition="top center",
    marker=dict(size=5), name="orig center"
))
for r in report_rows:
    cx, cy, cz = r["center_world_mm"]
    fig.add_trace(go.Scatter3d(
        x=[cx], y=[cy], z=[cz],
        mode="markers+text", text=[f"copy{r['copy_idx']}"], textposition="top center",
        marker=dict(size=5), name=f"copy{r['copy_idx']} center"
    ))

# Optional CT volume
if show_ct_volume:
    z_slice = slice(None, None, downsample)
    y_slice = slice(None, None, downsample)
    x_slice = slice(None, None, downsample)
    vol = out_img[z_slice, y_slice, x_slice]
    p1, p99 = np.percentile(vol, [1, 99])
    vol = np.clip(vol, p1, p99); vol = (vol - p1) / max(p99 - p1, 1e-6)
    zz, yy, xx = np.mgrid[
        0:vol.shape[0]*spacing[0]:spacing[0],
        0:vol.shape[1]*spacing[1]:spacing[1],
        0:vol.shape[2]*spacing[2]:spacing[2],
    ]
    fig.add_trace(go.Volume(
        x=xx.flatten(), y=yy.flatten(), z=zz.flatten(),
        value=vol.astype(np.float32).flatten(),
        opacity=0.08, surface_count=10, name="CT volume",
        opacityscale=[[0, 0.0], [1, 1.0]], showscale=False
    ))

fig.update_layout(
    scene=dict(
        xaxis_title="X (mm)", yaxis_title="Y (mm)", zaxis_title="Z (mm)",
        aspectmode="data",
    ),
    legend=dict(itemsizing="constant"),
    margin=dict(l=0, r=0, t=35, b=0),
    title="3D Copy-Paste Tumor in Same Case — Liver (gray), Original (blue), Pasted (red)"
)
fig.show()



In [ ]:
# Save augmented CT
nib.save(
    nib.Nifti1Image(out_img.astype(np.float32), img_nii.affine, img_nii.header),
    "aug_ct_samecase_70_1.nii.gz"
)

# Save augmented segmentation
nib.save(
    nib.Nifti1Image(out_lab.astype(np.int16), lab_nii.affine, lab_nii.header),
    "aug_seg_samecase_70_1.nii.gz"
)


In [ ]:
import os, re, glob
import nibabel as nib
import numpy as np
from skimage import measure
from scipy import ndimage as ndi

# =========================
# CONFIG
# =========================
DATA_DIR = "Data"
DATA_AUG_DIR = "Data_aug"

IMAGE_DIR = os.path.join(DATA_DIR, "image")
LABEL_DIR = os.path.join(DATA_DIR, "labels")

OUT_IMAGE_DIR = os.path.join(DATA_AUG_DIR, "image")
OUT_LABEL_DIR = os.path.join(DATA_AUG_DIR, "labels")

# Augmentation parameters (same as your single-case code)
LIVER_LABEL = 1
TUMOR_LABEL = 2
NUM_COPIES = 1                 # how many new tumors to paste per case
BLEND_BORDER = 0               # 0 = hard paste (set >0 for feathered edges)
INTENSITY_SCALE_RANGE = (0.95, 1.05)
INTENSITY_SHIFT_RANGE = (-5.0, 5.0)
MIN_LIVER_COVERAGE = 0.85
AVOID_OVERLAP_WITH_ORIG = True
# Keep pasted tumors away from existing tumors & each other
OCCUPIED_CLEARANCE_VOX = 2    # dilated safety margin (voxels) around occupied areas
MIN_CENTER_SEPARATION_VOX = 12  # optional extra spacing between centers
RNG_SEED = None                  # None => non-deterministic; or set an int for reproducible but varied runs

# =========================
# HELPERS
# =========================
def ensure_dirs():
    os.makedirs(OUT_IMAGE_DIR, exist_ok=True)
    os.makedirs(OUT_LABEL_DIR, exist_ok=True)

def bbox_of_mask(mask, pad=2):
    zz, yy, xx = np.where(mask)
    zmin, zmax = max(0, zz.min()-pad), min(mask.shape[0]-1, zz.max()+pad)
    ymin, ymax = max(0, yy.min()-pad), min(mask.shape[1]-1, yy.max()+pad)
    xmin, xmax = max(0, xx.min()-pad), min(mask.shape[2]-1, xx.max()+pad)
    return (slice(zmin, zmax+1), slice(ymin, ymax+1), slice(xmin, xmax+1))

def feather_alpha(binary_mask, blend_border=3):
    if blend_border <= 0:
        return binary_mask.astype(np.float32)
    dist_to_bg = ndi.distance_transform_edt(binary_mask)
    alpha = np.clip(dist_to_bg / float(blend_border), 0.0, 1.0).astype(np.float32)
    return alpha

def centroid_world(mask, affine):
    if not mask.any():
        return (np.nan, np.nan, np.nan)
    coords = np.column_stack(np.where(mask))
    c_vox = coords.mean(axis=0)
    c_h = np.r_[c_vox, 1.0]
    c_w = (affine @ c_h)[:3]
    return tuple(map(float, c_w))

def try_random_centers_shuffled(liver_mask, patch_shape, rng, max_tries=4000):
    pz, py, px = patch_shape
    hz, hy, hx = pz//2, py//2, px//2
    Z, Y, X = liver_mask.shape
    zmin, zmax = hz, Z - (pz - hz)
    ymin, ymax = hy, Y - (py - hy)
    xmin, xmax = hx, X - (px - hx)
    if zmax <= zmin or ymax <= ymin or xmax <= xmin:
        return
    zz, yy, xx = np.where(liver_mask)
    keep = (zz >= zmin) & (zz < zmax) & (yy >= ymin) & (yy < ymax) & (xx >= xmin) & (xx < xmax)
    cand = np.column_stack([zz[keep], yy[keep], xx[keep]])
    if len(cand) == 0:
        return
    for idx in rng.permutation(len(cand))[:max_tries]:
        cz, cy, cx = cand[idx]
        yield int(cz), int(cy), int(cx)

def load_case_safely(image_path, label_path):
    img_nii = nib.load(image_path)
    lab_nii = nib.load(label_path)

    img_do = img_nii.dataobj
    lab_do = lab_nii.dataobj

    # ---- CT/MRI (slice first, then cast)
    if getattr(img_do, "ndim", None) == 4:
        if img_do.shape[0] <= 4:     # (C,Z,Y,X)
            img = np.asarray(img_do[0, ...], dtype=np.float32, order="C")
        else:                        # (Z,Y,X,C)
            img = np.asarray(img_do[..., 0], dtype=np.float32, order="C")
    else:
        img = np.asarray(img_do, dtype=np.float32, order="C")

    # ---- Label
    if getattr(lab_do, "ndim", None) == 4:
        if lab_do.shape[0] <= 4:
            lab = np.asarray(lab_do[0, ...], dtype=np.int16, order="C")
        else:
            lab = np.asarray(lab_do[..., 0], dtype=np.int16, order="C")
    else:
        lab = np.asarray(lab_do, dtype=np.int16, order="C")

    affine_img = img_nii.affine
    affine_lab = lab_nii.affine          # usually same; keep both to be safe
    img_header = img_nii.header.copy()
    lab_header = lab_nii.header.copy()

    zooms = img_header.get_zooms()
    spacing = np.array(zooms[:3], dtype=float)
    voxel_mm3 = float(np.prod(spacing))
    return img, lab, affine_img, affine_lab, img_header, lab_header, spacing, voxel_mm3



# =========================
# CORE AUGMENTATION (one case)
# =========================
def augment_case(image_path, label_path, out_image_path, out_label_path):
    
    (img, lab,
    affine_img, affine_lab,
    img_hdr, lab_hdr,
    spacing, voxel_mm3) = load_case_safely(image_path, label_path)
    
    # Sanity checks (will print once per case)
    # If you still see a 4D shape here, we failed to slice correctly.
    if img.ndim != 3: raise RuntimeError(f"CT not 3D: {img.shape}  -> check loader")
    if lab.ndim != 3: raise RuntimeError(f"Label not 3D: {lab.shape} -> check loader")


    # Masks
    liver_mask = (lab == LIVER_LABEL).astype(np.uint8)
    tumor_mask = (lab == TUMOR_LABEL).astype(np.uint8)

    if not liver_mask.any():
        print(f"[Skip] No liver in {os.path.basename(label_path)}")
        return False
    if not tumor_mask.any():
        print(f"[Skip] No tumor in {os.path.basename(label_path)}")
        return False

    '''
    # Pick a single tumor component (largest)
    lab_cc, n_cc = ndi.label(tumor_mask)
    sizes = [(i, (lab_cc == i).sum()) for i in range(1, n_cc+1)]
    if not sizes:
        raise ValueError("No connected components in tumor mask.")
    src_id = max(sizes, key=lambda t: t[1])[0]
    src_comp = (lab_cc == src_id)
    '''
    lab_cc, n_cc = ndi.label(tumor_mask)
    if n_cc == 0:
        raise ValueError("No tumor components in mask.")
    
    # --- Option 1: choose a random tumor (all components equally likely)
    src_id = np.random.default_rng(RNG_SEED if RNG_SEED is not None else None).integers(1, n_cc + 1)

    
    # --- Option 2 (alternative): choose proportional to size
    # sizes = [(i, (lab_cc == i).sum()) for i in range(1, n_cc+1)]
    # ids, weights = zip(*sizes)
    # src_id = rng.choice(ids, p=np.array(weights)/sum(weights))
    
    src_comp = (lab_cc == src_id)

    # Crop source tumor patch
    src_slz, src_sly, src_slx = bbox_of_mask(src_comp, pad=2)
    tumor_ct_patch = img[src_slz, src_sly, src_slx].astype(np.float32)
    tumor_mask_patch = src_comp[src_slz, src_sly, src_slx].astype(bool)
    alpha_patch = feather_alpha(tumor_mask_patch, BLEND_BORDER)

    # Distance transform from original tumor (for separation)
    if AVOID_OVERLAP_WITH_ORIG:
        dt_from_orig = ndi.distance_transform_edt(~tumor_mask)
    else:
        dt_from_orig = None

    # Output volumes
    out_img = img.copy()
    out_lab = lab.copy()
    pasted_masks = []
    report_rows = []
    # Start with original tumors as occupied
    occupied_mask = tumor_mask.copy()
    
    # Precompute a dilated "forbidden" zone around occupied voxels
    # (keeps patches away by OCCUPIED_CLEARANCE_VOX voxels)
    structure = ndi.generate_binary_structure(3, 1)  # 6-neighborhood
    forbidden = ndi.binary_dilation(
        occupied_mask,
        structure=structure,
        iterations=int(max(OCCUPIED_CLEARANCE_VOX, 0))
    )
    dist_from_occupied = ndi.distance_transform_edt(~occupied_mask)


    rng = np.random.default_rng(RNG_SEED)
    pasted_count = 0

    for k in range(NUM_COPIES):
        placed = False
        for center in try_random_centers_shuffled(liver_mask, tumor_mask_patch.shape, rng, max_tries=4000):
            pz, py, px = tumor_mask_patch.shape
            hz, hy, hx = pz//2, py//2, px//2
            cz, cy, cx = center
            z1, z2 = cz-hz, cz-hz+pz
            y1, y2 = cy-hy, cy-hy+py
            x1, x2 = cx-hx, cx-hx+px
    
            roi_liver = liver_mask[z1:z2, y1:y2, x1:x2]
            roi_forbidden = forbidden[z1:z2, y1:y2, x1:x2]
    
            # ---- HARD NO-OVERLAP RULE with occupied (+clearance)
            if (tumor_mask_patch & roi_forbidden).any():
                continue
    
            # ---- Coverage rule (after forbidding overlap)
            inside_frac = (tumor_mask_patch & roi_liver).sum() / (tumor_mask_patch.sum() + 1e-6)
            if inside_frac < MIN_LIVER_COVERAGE:
                continue
    
            # ---- Optional: center separation (vs. original+pastes)
            if MIN_CENTER_SEPARATION_VOX > 0:
                if dist_from_occupied[cz, cy, cx] < MIN_CENTER_SEPARATION_VOX:
                    continue
    
            # ---- Paste (intensity jitter + feather)
            scale = rng.uniform(*INTENSITY_SCALE_RANGE)
            shift = rng.uniform(*INTENSITY_SHIFT_RANGE)
            roi_ct = out_img[z1:z2, y1:y2, x1:x2]
            alpha_eff = alpha_patch * tumor_mask_patch.astype(np.float32)
            pasted_ct = (1.0 - alpha_eff) * roi_ct + alpha_eff * (tumor_ct_patch * scale + shift)
            out_img[z1:z2, y1:y2, x1:x2] = pasted_ct
    
            # ---- Labels
            new_mask_full = np.zeros_like(out_lab, dtype=bool)
            new_mask_full[z1:z2, y1:y2, x1:x2] = tumor_mask_patch
            out_lab[new_mask_full] = TUMOR_LABEL
            pasted_masks.append(new_mask_full)

            pasted_count += 1
    
            # ---- UPDATE OCCUPIED + FORBIDDEN + DISTANCE for next iterations
            occupied_mask |= new_mask_full
            forbidden = ndi.binary_dilation(
                occupied_mask,
                structure=structure,
                iterations=int(max(OCCUPIED_CLEARANCE_VOX, 0))
            )
            dist_from_occupied = ndi.distance_transform_edt(~occupied_mask)
    
            # ---- reporting (unchanged)
            vox = int(new_mask_full.sum())
            vol_mm3 = vox * voxel_mm3
            c_src = centroid_world(src_comp, affine)
            c_new = centroid_world(new_mask_full, affine)
            

            offset = tuple(float(b - a) for a, b in zip(c_src, c_new))
            report_rows.append({
                "copy_idx": k+1,
                "voxels": vox,
                "volume_mm3": vol_mm3,
                "center_world_mm": c_new,
                "offset_from_src_mm": offset,
                "liver_coverage": float(inside_frac),
                "scale": float(scale),
                "shift": float(shift),
            })
            placed = True
            break
    
        if not placed:
            print(f"[Warn] Could not place tumor copy #{k+1} (relax coverage or clearance).")


    # Save outputs using original headers/affine to keep geometry consistent
    nib.save(nib.Nifti1Image(out_img.astype(np.float32), affine_img, img_hdr), out_image_path)
    nib.save(nib.Nifti1Image(out_lab.astype(np.int16),  affine_lab, lab_hdr),  out_label_path)


    vox = int(src_comp.sum())
    vol_mm3 = vox * voxel_mm3
    c_src = centroid_world(src_comp, affine)
    print(f"[OK] {os.path.basename(image_path)}  pasted:{pasted_count}  "
                    f"orig_tumor_vox:{vox}  vol:{vol_mm3:.1f} mm³  "
                    f"center(mm):{tuple(round(v,1) for v in c_src)}")
    return True

# =========================
# BATCH DRIVER
# =========================
def main():
    ensure_dirs()

    # find all images like liver_XX_0000.nii.gz
    image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "liver_*_0000.nii.gz")))

    # pattern to extract the case id ("liver_1_0000.nii.gz" -> "liver_1")
    pat = re.compile(r"^(?P<prefix>liver_\d+)_0000\.nii\.gz$")

    total = 0
    ok = 0
    skipped = 0
    for img_path in image_files:
        fname = os.path.basename(img_path)
        m = pat.match(fname)
        if not m:
            print(f"[Skip] Unexpected image name: {fname}")
            skipped += 1
            continue
        base = m.group("prefix")  # e.g., liver_70
        lab_path = os.path.join(LABEL_DIR, f"{base}.nii.gz")
        if not os.path.exists(lab_path):
            print(f"[Skip] Missing label for {fname}: {lab_path}")
            skipped += 1
            continue

        out_img_path = os.path.join(OUT_IMAGE_DIR, f"{base}_0000.nii.gz")
        out_lab_path = os.path.join(OUT_LABEL_DIR,  f"{base}.nii.gz")

        # idempotent: skip if already exists
        if os.path.exists(out_img_path) and os.path.exists(out_lab_path):
            print(f"[Skip] Already exists: {base}")
            skipped += 1
            continue

        total += 1
        try:
            if augment_case(img_path, lab_path, out_img_path, out_lab_path):
                ok += 1
        except Exception as e:
            print(f"[Error] {base}: {e}")

    print(f"\nDone. processed={total}  ok={ok}  skipped={skipped}  out_dir={DATA_AUG_DIR}")

if __name__ == "__main__":
    main()
